In [5]:
import pandas as pd
import numpy as np
import re

In [2]:
df_asort = pd.read_parquet('dane/Asort.parquet')
df_pozd = pd.read_parquet("dane/Pozdok.parquet")
df_dok = pd.read_parquet("dane/Dok.parquet")
df_tow = pd.read_parquet("dane/Towar.parquet")

In [6]:
# ====================================================
# 1. PRAWIDŁOWA NORMALIZACJA POLSKICH ZNAKÓW (bez zjadania liter!)
# ====================================================

polish_map = str.maketrans({
    "ą": "a", "ć": "c", "ę": "e", "ł": "l",
    "ń": "n", "ó": "o", "ś": "s", "ź": "z", "ż": "z",
    "Ą": "A", "Ć": "C", "Ę": "E", "Ł": "L",
    "Ń": "N", "Ó": "O", "Ś": "S", "Ź": "Z", "Ż": "Z"
})

def clean_name(text: str):
    # if pd.isna(text):
    #     return ""
    text = str(text)
    text = text.translate(polish_map)         # zamiana PL → ASCII 1:1
    text = text.lower()
    text = re.sub(r"[^a-z0-9]+", " ", text)   # zamiana reszty na spację
    text = re.sub(r"\s+", " ", text).strip()  # redukcja wielokrotnych spacji
    return text

def process_df(df):
    df = df.copy()

    # dodanie kolumny clean_name
    df["CleanName"] = df["Nazwa"].astype(str).apply(clean_name)

    return df

# ====================================================
# UŻYCIE
# ====================================================
# df_result = process_df(df_xxx)


In [7]:
df_tow_clean = process_df(df_tow)

In [5]:
df_pozd['TypTowaru'].value_counts()

0    4256616
1      11768
3       2356
6       1698
2        218
Name: TypTowaru, dtype: int64

In [6]:
df_tow['Aktywny'].value_counts()

0    21524
1    19854
Name: Aktywny, dtype: int64

In [61]:
df_tow_clean.columns

Index(['TowId', 'AsId', 'JMId', 'KatId', 'Producent', 'ArtId', 'Nazwa',
       'Skrot', 'Kod', 'TypTowaru', 'Indeks1', 'Indeks2', 'Opis1', 'Opis2',
       'Opis3', 'Opis4', 'TermWazn', 'Marza', 'HurtRabat', 'NocNarzut',
       'OpcjaMarzy', 'OpcjaRabatu', 'OpcjaNarzutu', 'CenaEw', 'CenaDet',
       'CenaHurt', 'CenaNoc', 'CenaDod', 'CenaOtwarta', 'PoziomCen', 'PrefPLU',
       'Stawka', 'IleWZgrzewce', 'IleWCalosci', 'KodZgrzewki', 'Aktywny',
       'Waga', 'Szerokosc', 'Wysokosc', 'Glebokosc', 'CKU', 'BlokDostawcow',
       'BlokCenyZak', 'BlokCenSp', 'BlokZmian', 'Rezerwa1', 'Rezerwa2',
       'CentrTowId', 'Zmiana', 'Akcyzowy', 'SledzPartii', 'MaxCenaZak',
       'PrzeliczJM', 'NrDrukarki', 'Cena5', 'Cena6', 'ProgPromocji',
       'ZmianaIstotna', 'ZmianaTylkoCen', 'Przeznaczenie', 'ObslugaPartii',
       'UkrycNaKasie', 'KodCN', 'MinCenaSp', 'SubsysKoduGlownego', 'StatusZam',
       'KodGlownyCentralny', 'NowoscOd', 'NowoscPrzez',
       'WysylacNaSklepInternetowy', 'GrupaGTU', 'Kr

In [ ]:
# df_tow_clean
tow_cols = ['TowId', 'AsId', 'JMId', 'Nazwa', 'Kod', 'Opis1', 'Producent', 'Marza', 'Stawka', 'Aktywny', 'CleanName']
df_tow_selected = df_tow_clean[tow_cols].copy()
df_tow_selected = df_tow_selected.rename(columns={'Nazwa': 'NazwaTow', 'Aktywny': 'AktywnyTow', 'Kod': 'EAN'})

# df_asort
asort_cols = ['AsId', 'Nazwa']
df_asort_selected = df_asort[asort_cols].copy()
df_asort_selected = df_asort_selected.rename(columns={'Nazwa': 'NazwaAsort'})

# df_dok
dok_cols = ['DokId', 'Data', 'KolejnyWDniu', 'NrDok', 'TypDok', 'Aktywny', 'Razem', 'DoZaplaty', 'Zaplacono']
df_dok_selected = df_dok[dok_cols].copy()
df_dok_selected = df_dok_selected.rename(columns={'Aktywny': 'AktywnyDok'})

# df_pozd
pozd_cols = ['DokId', 'Kolejnosc', 'NrPozycji', 'TowId', 'TypPoz', 'IloscPlus', 'IloscMinus', 'CenaPrzedRab', 'CenaPoRab', 'Wartosc', 'CenaDet']
df_pozd_selected = df_pozd[pozd_cols].copy()

print(f"df_tow_selected:   {df_tow_selected.shape}")
print(f"df_asort_selected: {df_asort_selected.shape}")
print(f"df_dok_selected:   {df_dok_selected.shape}")
print(f"df_pozd_selected:   {df_pozd_selected.shape}")

df_tow_selected:   (41378, 11)
df_asort_selected: (337, 2)
df_dok_selected:   (952122, 9)
df_pozd_selected:   (4272656, 11)


In [9]:
df_tow_selected.to_parquet("dane/interim/Towar-columns_selected-records_full.parquet", compression='zstd', index=False)
df_asort_selected.to_parquet("dane/interim/Asort-columns_selected-records_full.parquet", compression='zstd', index=False)
df_dok_selected.to_parquet("dane/interim/Dok-columns_selected-records_full.parquet", compression='zstd', index=False)
df_pozd_selected.to_parquet("dane/interim/PozDok-columns_selected-records_full.parquet", compression='zstd', index=False)

# parametry dla kompresji przy zapisie .parquet
# snappy  — najszybszy odczyt, słabsza kompresja - domyślny 
# gzip    — dobry balans
# zstd    — bardzo dobry balans (szybkość + kompresja)
# brotli  — najmniejszy plik, wolniejszy odczyt